# 04 Predictive Modeling

## Objective

Test whether non-disease county-level indicators can predict age-adjusted premature mortality across U.S. counties.

This model intentionally excludes disease-burden indicators such as coronary heart disease, stroke, COPD, diabetes, and high blood pressure because those variables are outcome-adjacent. The goal is to focus on more actionable health behavior, health-status, and socioeconomic predictors.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance


In [ ]:
data_path = Path('../data/processed/mortality_places_acs_merged_2015_2020.csv')
df = pd.read_csv(data_path, dtype={'county_fips': str})
df.shape


In [ ]:
target = 'premature_mortality_age_adjusted_rate_2015_2019'

features = [
    'poor_physical_health_rate',
    'poor_mental_health_rate',
    'physical_inactivity_rate',
    'smoking_rate',
    'poverty_rate',
    'median_household_income',
    'bachelor_plus_rate',
    'unemployment_rate',
    'insufficient_sleep_rate',
    'obesity_rate',
    'uninsured_rate'
]

label_map = {
    'poor_physical_health_rate': 'Poor Physical Health',
    'poor_mental_health_rate': 'Poor Mental Health',
    'physical_inactivity_rate': 'Physical Inactivity',
    'smoking_rate': 'Smoking',
    'poverty_rate': 'Poverty',
    'median_household_income': 'Median Household Income',
    'bachelor_plus_rate': "Bachelor's Degree or Higher",
    'unemployment_rate': 'Unemployment',
    'insufficient_sleep_rate': 'Insufficient Sleep',
    'obesity_rate': 'Obesity',
    'uninsured_rate': 'Uninsured'
}

model_df = df[[target] + features].dropna().copy()
model_df.shape


In [ ]:
X = model_df[features]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

linear_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

rf_model = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    min_samples_leaf=5,
    n_jobs=-1
)


In [ ]:
models = {
    'Linear Regression': linear_model,
    'Random Forest': rf_model
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    results.append({
        'model': name,
        'test_r2': r2_score(y_test, preds),
        'test_mae': mean_absolute_error(y_test, preds),
        'test_rmse': np.sqrt(mean_squared_error(y_test, preds))
    })

results_df = pd.DataFrame(results)
results_df


## Model Performance

The Random Forest model performed better than the linear regression model, suggesting that nonlinear relationships or interactions may help explain county-level premature mortality patterns.

In [ ]:
linear_coefs = linear_model.named_steps['model'].coef_

linear_importance = pd.DataFrame({
    'feature': features,
    'label': [label_map[f] for f in features],
    'linear_standardized_coefficient': linear_coefs,
    'abs_linear_standardized_coefficient': np.abs(linear_coefs)
}).sort_values('abs_linear_standardized_coefficient', ascending=False)

linear_importance


In [ ]:
perm = permutation_importance(
    rf_model,
    X_test,
    y_test,
    n_repeats=20,
    random_state=42,
    n_jobs=-1
)

rf_importance = pd.DataFrame({
    'feature': features,
    'label': [label_map[f] for f in features],
    'rf_permutation_importance': perm.importances_mean,
    'rf_permutation_importance_std': perm.importances_std
}).sort_values('rf_permutation_importance', ascending=False)

rf_importance


## Modeling Takeaway

Using non-disease predictors only, the Random Forest model explained roughly 72% of test-set variation in county-level premature mortality. Poor mental health rate was the strongest predictor by permutation importance, followed by physical inactivity, uninsured rate, poverty, educational attainment, smoking, and income.

This does not prove causation. It suggests that premature mortality is tied to a broader county-level risk profile involving mental health, physical inactivity, healthcare access, socioeconomic disadvantage, and smoking.